CONFIGURACIÓN

In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
REPORTS_DIR = PROJECT_ROOT / "reports"
EVIDENCE_DIR = PROJECT_ROOT / "docs" / "evidence" / "profiling"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

YEARS = [2015, 2016, 2017, 2018, 2019]

def save_table(df, name, show=True):
    """Guarda una tabla de perfilado como CSV (evidencia) y la muestra."""
    path = EVIDENCE_DIR / f"{name}.csv"
    df.to_csv(path, index=True, encoding="utf-8")
    if show:
        display(df)
    print(f"-> saved {path.relative_to(PROJECT_ROOT)}")

print("Project root:", PROJECT_ROOT)

Project root: c:\WORKSHOP 3


CARGA DE LOS 5 ARCHIVOS

In [2]:
dfs = {}
for y in YEARS:
    path = RAW_DIR / f"{y}.csv"
    assert path.exists(), f"No se encontró {path}"
    dfs[y] = pd.read_csv(path, encoding="utf-8-sig")

for y, df in dfs.items():
    print(f"{y}: {df.shape[0]} filas x {df.shape[1]} columnas")
    print("   ", list(df.columns))

2015: 158 filas x 12 columnas
    ['Country', 'Region', 'Happiness Rank', 'Happiness Score', 'Standard Error', 'Economy (GDP per Capita)', 'Family', 'Health (Life Expectancy)', 'Freedom', 'Trust (Government Corruption)', 'Generosity', 'Dystopia Residual']
2016: 157 filas x 13 columnas
    ['Country', 'Region', 'Happiness Rank', 'Happiness Score', 'Lower Confidence Interval', 'Upper Confidence Interval', 'Economy (GDP per Capita)', 'Family', 'Health (Life Expectancy)', 'Freedom', 'Trust (Government Corruption)', 'Generosity', 'Dystopia Residual']
2017: 155 filas x 12 columnas
    ['Country', 'Happiness.Rank', 'Happiness.Score', 'Whisker.high', 'Whisker.low', 'Economy..GDP.per.Capita.', 'Family', 'Health..Life.Expectancy.', 'Freedom', 'Generosity', 'Trust..Government.Corruption.', 'Dystopia.Residual']
2018: 156 filas x 9 columnas
    ['Overall rank', 'Country or region', 'Score', 'GDP per capita', 'Social support', 'Healthy life expectancy', 'Freedom to make life choices', 'Generosity', 

DIFERENCIAS DE ESQUEMA (NOMBRES Y TIPOS)

In [3]:
# Matriz de presencia de columnas por año
all_cols = sorted(set().union(*[set(df.columns) for df in dfs.values()]))
schema = pd.DataFrame(
    {y: ["X" if c in df.columns else "-" for c in all_cols] for y, df in dfs.items()},
    index=all_cols,
)
save_table(schema, "01_schema_presence")

# Tipos de dato por año
dtypes = pd.DataFrame({y: df.dtypes.astype(str) for y, df in dfs.items()}).fillna("-")
save_table(dtypes, "01_schema_dtypes")

,2015,2016,2017,2018,2019
Country,X,X,X,-,-
Country or region,-,-,-,X,X
Dystopia Residual,X,X,-,-,-
Dystopia.Residual,-,-,X,-,-
Economy (GDP per Capita),X,X,-,-,-
Economy..GDP.per.Capita.,-,-,X,-,-
Family,X,X,X,-,-
Freedom,X,X,X,-,-
Freedom to make life choices,-,-,-,X,X
GDP per capita,-,-,-,X,X


-> saved docs\evidence\profiling\01_schema_presence.csv


,2015,2016,2017,2018,2019
Country,object,object,object,-,-
Country or region,-,-,-,object,object
Dystopia Residual,float64,float64,-,-,-
Dystopia.Residual,-,-,float64,-,-
Economy (GDP per Capita),float64,float64,-,-,-
Economy..GDP.per.Capita.,-,-,float64,-,-
Family,float64,float64,float64,-,-
Freedom,float64,float64,float64,-,-
Freedom to make life choices,-,-,-,float64,float64
GDP per capita,-,-,-,float64,float64


-> saved docs\evidence\profiling\01_schema_dtypes.csv


REPORTES AUTOMÁTICOS HTML

In [4]:
from ydata_profiling import ProfileReport

for y, df in dfs.items():
    report = ProfileReport(df, title=f"World Happiness {y} — Data Profiling")
    out = REPORTS_DIR / f"happiness_{y}_profiling.html"
    report.to_file(out)
    print("-> saved", out.relative_to(PROJECT_ROOT))

C:\Users\monte\AppData\Local\Temp\ipykernel_13532\753861115.py:1: DeprecationWarning: 
    `import ydata_profiling` is deprecated and will not receive more updates. 
    Please install fg-data-profiling via `pip install fg-data-profiling` and use `import data_profiling` instead.
    
  from ydata_profiling import ProfileReport


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 12/12 [00:00<?, ?it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

... (21 outputs left out)

Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

-> saved reports\happiness_2019_profiling.html
